In [1]:
from pathlib import Path
import os
import pandas as pd
from ultralytics import YOLO

PROJECT_ROOT = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection")
os.chdir(PROJECT_ROOT)

MODEL_PATHS = {
    "archive1_classification": PROJECT_ROOT / "models" / "classification" / "archive1_best.pt",
    "archive2_severity": PROJECT_ROOT / "models" / "classification" / "archive2_best.pt",
    "archive5_classification": PROJECT_ROOT / "models" / "classification" / "archive5_best.pt",

    "cardd_coco_damage_detection": PROJECT_ROOT / "models" / "detection" / "cardd_coco_best.pt",
    "damaged_parts_support": PROJECT_ROOT / "models" / "detection" / "damaged_parts_best.pt",

    "archive4_segmentation": PROJECT_ROOT / "models" / "segmentation" / "archive4_best.pt",
    "cardd_sod_damage_segmentation": PROJECT_ROOT / "models" / "segmentation" / "cardd_sod_best.pt",
    "carparts_segmentation": PROJECT_ROOT / "models" / "segmentation" / "carparts_best.pt",
}

rows = []

for model_name, model_path in MODEL_PATHS.items():
    if not model_path.exists():
        rows.append({
            "model": model_name,
            "exists": False,
            "task": "missing",
            "classes": "missing"
        })
        continue

    model = YOLO(str(model_path))

    if isinstance(model.names, dict):
        class_names = ", ".join([str(v) for v in model.names.values()])
    else:
        class_names = ", ".join([str(v) for v in model.names])

    rows.append({
        "model": model_name,
        "exists": True,
        "task": model.task,
        "classes": class_names
    })

inventory_df = pd.DataFrame(rows)
inventory_df

,model,exists,task,classes
0,archive1_classification,True,classify,"00-damage, 01-whole"
1,archive2_severity,True,classify,"01-minor, 02-moderate, 03-severe"
2,archive5_classification,True,classify,"1, 2, 3, 4, 5, 6"
3,cardd_coco_damage_detection,True,detect,"dent, scratch, crack, glass shatter, lamp brok..."
4,damaged_parts_support,True,detect,"front_bumper_damage, rear_bumper_damage, hood_..."
5,archive4_segmentation,True,segment,"be_den, mat_bo_phan, mop_lom, rach, thung, tra..."
6,cardd_sod_damage_segmentation,True,segment,damage
7,carparts_segmentation,True,segment,"back_bumper, back_door, back_glass, back_left_..."


In [2]:
from pathlib import Path
from ultralytics import YOLO

IMAGE_PATH = PROJECT_ROOT / "test_images" / "sample_car.jpg"

print("Image exists:", IMAGE_PATH.exists())

for model_name, model_path in MODEL_PATHS.items():
    if not model_path.exists():
        print(f"\n{model_name}: model missing")
        continue

    print(f"\n========== {model_name} ==========")

    model = YOLO(str(model_path))

    if model.task == "classify":
        result = model.predict(source=str(IMAGE_PATH), verbose=False)[0]

        top1_id = int(result.probs.top1)
        top1_conf = float(result.probs.top1conf)

        print("Prediction:", model.names[top1_id])
        print("Confidence:", round(top1_conf, 4))

    else:
        result = model.predict(
            source=str(IMAGE_PATH),
            conf=0.10,
            imgsz=640,
            verbose=False
        )[0]

        if result.boxes is None or len(result.boxes) == 0:
            print("No detections")
            continue

        for box in result.boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            print(model.names[class_id], round(confidence, 4))

        if model.task == "segment":
            print("Masks returned:", result.masks is not None)

Image exists: True

========== archive1_classification ==========
Prediction: 00-damage
Confidence: 1.0

========== archive2_severity ==========
Prediction: 03-severe
Confidence: 0.9413

========== archive5_classification ==========
Prediction: 6
Confidence: 0.9147

========== cardd_coco_damage_detection ==========
dent 0.8657

========== damaged_parts_support ==========
front_bumper_damage 0.1385

========== archive4_segmentation ==========
mop_lom 0.6642
Masks returned: True

========== cardd_sod_damage_segmentation ==========
damage 0.863
Masks returned: True

========== carparts_segmentation ==========
wheel 0.6523
back_bumper 0.2025
Masks returned: True
